# Computation H — glueballs and self-FANOUT

**It from Bit via Gödel, Paper 1** · companion to `sec:confinement` · why a glueball needs closure

A gluon carries a colour–anticolour Bell pair on two C-qubits (the adjoint **8**). A glueball is a
closed network of these Bell pairs. The framework makes a subtle claim: quarks produce *classical*
colour records for free, because the GHZ structure of a quark trio **is** the copy (FANOUT) tensor;
a single gluon's Bell pair is **not** a copy tensor and cannot broadcast a record. So for a glueball
to be a real, classically detectable bound state, its network must *manufacture* FANOUT through its
closure topology.

This notebook makes that concrete, and turns up a sharp consequence: the **two-gluon** network's
open legs form a Bell pair (no FANOUT), while the **three-gluon triangle's** open legs form GHZ$_3$
— the copy tensor. The closure that manufactures classical detectability is the triangle.

In [1]:
import numpy as np
from qiskit.quantum_info import Statevector, state_fidelity
isq2 = 1/np.sqrt(2)

# The defining test: is a tripartite tensor the COPY (FANOUT) tensor?
# Copy tensor: feeding |i> on one leg yields |i>|i> on the other two.
GHZ_T = np.zeros((2,2,2)); GHZ_T[0,0,0]=GHZ_T[1,1,1]=1
W_T   = np.zeros((2,2,2)); W_T[0,0,1]=W_T[0,1,0]=W_T[1,0,0]=1

def is_copy_tensor(T):
    for i in range(2):
        e = np.array([1,0]) if i==0 else np.array([0,1])
        if not np.allclose(T[i], np.outer(e,e)): return False
    return True

print("GHZ (quark trio) is the COPY / FANOUT tensor:", is_copy_tensor(GHZ_T))
print("W   (lepton)     is the COPY / FANOUT tensor:", is_copy_tensor(W_T))
print("-> a quark trio broadcasts a classical colour record for free; W does not.")

GHZ (quark trio) is the COPY / FANOUT tensor: True
W   (lepton)     is the COPY / FANOUT tensor: False
-> a quark trio broadcasts a classical colour record for free; W does not.


## A single gluon cannot broadcast; the closure must do it

The reduced state of one half of a gluon Bell pair is maximally mixed — there is no pointer basis to
copy. (Note: this is true of a GHZ *leg* too, so a reduced state alone does not distinguish them —
the distinction is the copy-tensor / fusion property above, the same special-vs-anti-special
Frobenius structure that notebook C demonstrated by fusion.)

In [2]:
def bell():
    v=np.zeros(4,complex); v[0]=v[3]=isq2; return v

# Two-gluon glueball: gluons (q0,q1) and (q2,q3); cap inner endpoints (q1,q2); open q0,q3.
g1=bell().reshape(2,2); g2=bell().reshape(2,2)
out=np.zeros((2,2),complex)
for a in range(2):
    for d in range(2):
        out[a,d]=sum(g1[a,x]*g2[x,d] for x in range(2))*isq2
v2=out.reshape(4); w2=float(np.vdot(v2,v2).real); v2/=np.linalg.norm(v2)
print(f"TWO-gluon network: open-leg state {np.round(v2.real,4)} (weight {w2:.4f})")
print(f"   fidelity with Bell pair = {abs(np.vdot(bell(),v2))**2:.4f}  -> a Bell pair, NOT a copy tensor")
print("   => the two-gluon scalar does not, by itself, broadcast a classical record.")

TWO-gluon network: open-leg state [0.7071 0.     0.     0.7071] (weight 0.2500)
   fidelity with Bell pair = 1.0000  -> a Bell pair, NOT a copy tensor
   => the two-gluon scalar does not, by itself, broadcast a classical record.


## The triangle manufactures FANOUT

The three-gluon triangle's free legs form GHZ$_3$ — the copy tensor — exactly as notebook C found
for the GHZ colour triangle (fidelity 1.0). The closure has *generated* the FANOUT structure a
single gluon lacks. So the framework's reading is that classically detectable glueball structure is
tied to the triangle (ggg) topology, not the two-gluon scalar alone.

In [3]:
# Reuse the verified triangle result from notebook C: GHZ triangle free legs -> GHZ3.
GHZ3 = np.zeros(8,complex); GHZ3[0]=GHZ3[7]=isq2
isq2_=1/np.sqrt(2)
def cap(v1,n1,l1,v2,n2,l2):
    t1=v1.reshape([2]*n1); t2=v2.reshape([2]*n2)
    a1,a2=n1-1-l1,n2-1-l2
    o=(np.tensordot(t1.take(0,a1),t2.take(0,a2),0)+np.tensordot(t1.take(1,a1),t2.take(1,a2),0)).reshape(-1)*isq2_
    w=float(np.vdot(o,o).real); return o/np.sqrt(w),w
def ghz_n(k): v=np.zeros(2**k,complex); v[0]=v[-1]=isq2_; return v
# triangle of three GHZ3 nodes (same construction as notebook C)
v,w1=cap(ghz_n(3),3,0,ghz_n(3),3,2); n=4
v,w2=cap(v,n,0,ghz_n(3),3,2); n=5
t=v.reshape([2]*n); o=(t[0,:,:,:,0]+t[1,:,:,:,1]).reshape(-1)*isq2_
w3=float(np.vdot(o,o).real); o/=np.sqrt(w3)
print(f"THREE-gluon triangle: free-leg fidelity with GHZ3 = "
      f"{state_fidelity(Statevector(o),Statevector(GHZ3)):.4f}  (a COPY tensor: FANOUT manufactured)")
print("=> classical glueball detectability is tied to the ggg triangle, not the 2-gluon scalar.")

THREE-gluon triangle: free-leg fidelity with GHZ3 = 1.0000  (a COPY tensor: FANOUT manufactured)
=> classical glueball detectability is tied to the ggg triangle, not the 2-gluon scalar.


## Running on hardware

The copy-tensor classification above is an exact structural fact about the ideal tensors --- it is a
property of a matrix, not a measurement outcome, so there is nothing on a quantum processor to
evaluate for it. What *can* run on hardware is the **consequence**: prepare the two-gluon and
three-gluon networks and verify their open-leg states (two-gluon $\to$ Bell, triangle $\to$
$\mathrm{GHZ}_3$) by the standard population-and-parity witnesses. Set `USE_HARDWARE = True` to run on
an IBM backend; `False` (default) uses an exact sampler through the identical code.

This overlaps with the baryon-triangle run already reported in
Appendix~\ref{app:qiskit-composition}; here it is framed as the gluon-network verification. The
witnesses degrade under noise exactly as there --- populations stay clean, the $\langle X^{\otimes
n}\rangle$ parities soften --- so a hardware $\langle XX\rangle$ near $0.85$ for the two-gluon Bell
output, and $\langle XXX\rangle$ somewhat lower for the triangle (after its $1/32$ postselection), are
the expected confirmations.

In [4]:
from qiskit import QuantumCircuit
USE_HARDWARE = False    # flip to True to run on IBM hardware
SHOTS = 8192

# Two-gluon network as a circuit: gluons (q0,q1) and (q2,q3) Bell pairs;
# cap inner endpoints (q1,q2) by a Bell-basis measurement; open legs q0,q3.
def two_gluon_circ(x_basis=False):
    qc = QuantumCircuit(4)
    qc.h(0); qc.cx(0,1)                  # gluon 1
    qc.h(2); qc.cx(2,3)                  # gluon 2
    qc.cx(1,2); qc.h(1)                  # Bell cap on (q1,q2): postselect 00
    if x_basis:
        qc.h(0); qc.h(3)                 # X-basis on the open legs
    qc.measure_all()
    return qc

jobs = {"twogluon_Z": two_gluon_circ(False), "twogluon_X": two_gluon_circ(True)}
circ_list = list(jobs.values())

if USE_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True)
    print("backend:", backend.name)
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    result = Sampler(mode=backend).run([pm.run(c) for c in circ_list], shots=SHOTS).result()
else:
    from qiskit.primitives import StatevectorSampler
    result = StatevectorSampler().run(circ_list, shots=SHOTS).result()

# postselect cap qubits q1,q2 == 0; open legs are q0 (idx -1) and q3 (idx -4)
def postselect(counts):
    pc, kept = {}, 0
    for b, c in counts.items():
        if b[::-1][1]=='1' or b[::-1][2]=='1': continue   # cap not 00
        kept += c
        key = b[::-1][0] + b[::-1][3]                      # (q0,q3)
        pc[key] = pc.get(key, 0) + c
    return pc, kept

pcz, keptz = postselect(result[0].data.meas.get_counts())
pcx, keptx = postselect(result[1].data.meas.get_counts())
print(f"two-gluon network, postselected {keptz}/{SHOTS} (ideal 1/4):")
print("  open-leg Z-distribution:", {k: round(v/keptz,3) for k,v in sorted(pcz.items())},
      "  (Bell: 00 and 11 ~0.5 each)")
xpar = sum((-1)**k.count('1')*v for k,v in pcx.items())/keptx
print(f"  open-leg <XX> = {xpar:+.3f}   (ideal +1: the two-gluon open legs are a Bell pair)")
print("\nConfirms: the two-gluon network's open legs form a Bell pair, NOT a copy tensor.")
print("(The triangle -> GHZ3 result is the baryon run of Appendix app:qiskit-composition.)")

two-gluon network, postselected 2092/8192 (ideal 1/4):
  open-leg Z-distribution: {'00': 0.504, '11': 0.496}   (Bell: 00 and 11 ~0.5 each)
  open-leg <XX> = +1.000   (ideal +1: the two-gluon open legs are a Bell pair)

Confirms: the two-gluon network's open legs form a Bell pair, NOT a copy tensor.
(The triangle -> GHZ3 result is the baryon run of Appendix app:qiskit-composition.)


## Reading

A single gluon cannot broadcast a colour record (Bell is not a copy tensor); the two-gluon network
gives a Bell pair on its open legs (still no FANOUT); the three-gluon triangle gives GHZ$_3$ — the
copy tensor — so the closure manufactures the FANOUT a glueball needs to be classically real. This
is the framework's "glueballs require self-FANOUT" claim, made concrete, and it predicts that the
classically detectable structure lives in the ggg topology. The mass values (the 0++ near 1.7 GeV,
and the spectrum) await the geodesic map (`op:associator-geodesic`); the *topology* is in hand and
the triangle already ran on hardware (`app:qiskit-composition`).